In [ ]:
import pandas as pd
import numpy as np
from pymatgen.core.composition import Composition
from periodictable import elements as pte

#  Convert a chemical formula into a vector of stoichiometric fractions.
  #  Example: Bi2Se3 → [0,0,0,...,Bi=0.4, Se=0.6,...]

element_symbols = [e.symbol for e in list(pte) if e.number <= 118]

def formula_to_vector(formula, element_symbols=element_symbols):
  
    vec = np.zeros(len(element_symbols))
    try:
        comp = Composition(formula)
        el_amt = comp.get_el_amt_dict()
        total = sum(el_amt.values())
        for el, amt in el_amt.items():
            if el in element_symbols:
                idx = element_symbols.index(el)
                vec[idx] = amt / total
    except Exception as e:
        print(f"⚠️ Could not parse {formula}: {e}")
    return vec

def add_composition_vectors(csv_file, output_file):
    df = pd.read_csv(csv_file)

   
    if "formula" not in df.columns:
        raise ValueError(f"'formula' column not found in {csv_file}")

    
    df["composition_vector"] = df["formula"].apply(lambda f: formula_to_vector(f).tolist())

   
    df.to_csv(output_file, index=False)
    print(f"✅ Added composition vectors and saved to {output_file}")
    return df



topo_df = add_composition_vectors(
    "materiae_AllTopological_withSOC.csv",
    "materiae_AllTopological_withSOC_withCompositionVectors.csv"
)

triv_df = add_composition_vectors(
    "materiae_filtered_by_spacegroups_Trivial_Insulators_NonTrivial_SI_negativeLabels.csv",
    "materiae_Trivial_withSOC_withCompositionVectors.csv"
)


✅ Added composition vectors and saved to materiae_AllTopological_withSOC_withCompositionVectors.csv
✅ Added composition vectors and saved to materiae_Trivial_withSOC_withCompositionVectors.csv


In [ ]:
# Apply to the discovery dataset

trivial_si_df = add_composition_vectors(
    "materiae_filtered_by_spacegroups_Trivial_Insulators_Trivial_SI.csv",
    "materiae_filtered_by_spacegroups_Trivial_Insulators_Trivial_SI_withCompositionVectors.csv"
)

✅ Added composition vectors and saved to materiae_filtered_by_spacegroups_Trivial_Insulators_Trivial_SI_withCompositionVectors.csv


In [ ]:
#merge the final positive and negative labelled datasets into one file
import pandas as pd


df_topo = pd.read_csv("materiae_AllTopological_withSOC_withCompositionVectors.csv")
df_triv = pd.read_csv("materiae_Trivial_withSOC_withCompositionVectors.csv")

df_topo["is_topological"] = 1
df_triv["is_topological"] = 0

df_all = pd.concat([df_topo, df_triv], ignore_index=True)

df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

output_file = "materiae_All_Labelled_withSOC_with_Composition_Vectors.csv"
df_all.to_csv(output_file, index=False)

print(f"✅ Combined dataset saved to: {output_file}")
print(f"Total materials: {len(df_all)} (Topological: {df_topo.shape[0]}, Trivial: {df_triv.shape[0]})")


✅ Combined dataset saved to: materiae_All_Labelled_withSOC_with_Composition_Vectors.csv
Total materials: 22245 (Topological: 8082, Trivial: 14163)
